In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Configuration
VOLUME_BASE = "/Volumes/imdb_final_project/raw/raw_store"
ENABLE_SCHEMA_EVOLUTION = False



# Helper function to reduce code duplication
def create_bronze_table(subfolder, schema_hints,delimiter="\t"):
    """Helper to create bronze ingestion logic"""
    reader = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.schemaLocation", f"{VOLUME_BASE}/{subfolder}/_schema_checkpoint")
            .option("cloudFiles.inferColumnTypes", "true")
            .option("cloudFiles.schemaHints", schema_hints)
            .option("delimiter", delimiter)
            .option("header", "true")
            .option("multiLine", "true")
            .option("escape", "\"")
            .option("nullValue", "\\N") 
    )
    
    if ENABLE_SCHEMA_EVOLUTION:
        reader = reader.option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    
    df = reader.load(f"{VOLUME_BASE}/{subfolder}/")
    
    # Rename columns
    for col_name in df.columns:
        clean_name = col_name.replace(" ", "_").replace("#", "Number")
        df = df.withColumnRenamed(col_name, clean_name)
    
    # Add audit columns
    return (
        df
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("ingestion_date", current_date())
    )


# BRONZE TABLES


@dlt.table(
    name="bronze_title_crew_raw",
    comment="Raw IMDB title.crew data - Directors and Writers"
)

def bronze_title_crew():
    return create_bronze_table("title_crew", "tconst STRING, directors STRING, writers STRING")
 
@dlt.table(
    name="bronze_title_basics_raw",
    comment="Raw IMDB title.basics data"
)

def bronze_title_basics():
    return create_bronze_table("title_basics", "tconst STRING, titleType STRING, primaryTitle STRING, originalTitle STRING, isAdult STRING, startYear STRING, endYear STRING, runtimeMinutes STRING, genres STRING")



The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-8986771962173388>, line 1
----> 1 import dlt
      2 from pyspark.sql.functions import *
      3 from pyspark.sql.types import *

ModuleNotFoundError: No module named 'dlt'

In [0]:
@dlt.table(
    name="silver_title_basics",
    comment="Cleaned title basics with type conversions and quality flags",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_all_or_drop({
    # PRIMARY KEY VALIDATION
    "valid_tconst": "tconst IS NOT NULL AND tconst RLIKE '^tt[0-9]{7,8}$'",
    
    # LOGICAL VALIDATION: startYear <= endYear (DROP invalid ranges)
    "start_before_end": "start_year <= end_year",
    
    # RUNTIME VALIDATION (from profiling: max seen was 12,000 min)
    "reasonable_runtime": "runtime_minutes < 50000"
})
@dlt.expect_all_or_fail({
    "tconst_not_null": "tconst IS NOT NULL"
})
def silver_title_basics():
    """Silver transformation for title.basics"""
    df = dlt.read_stream("bronze_title_basics_raw")
    
    # Replace \N with NULL
    df = df.select(
        *[when(col(c) == "\\N", None).otherwise(col(c)).alias(c) 
          for c in ["tconst", "titleType", "primaryTitle", "originalTitle", 
                    "isAdult", "startYear", "endYear", "runtimeMinutes", "genres"]],
        "ingestion_timestamp",
        "source_file",
        "ingestion_date"
    )
    
    # Handle NULL Values for genres
    df = df.withColumn("genres",
                      when(col("genres").isNull(), "Unknown")
                      .otherwise(col("genres")))
    
    # Type Conversions with NULL replacement
    df = (
        df
        # isAdult: STRING '0'/'1' → BOOLEAN
        .withColumn("is_adult",
                   when(col("isAdult") == "1", True)
                   .when(col("isAdult") == "0", False)
                   .otherwise(None).cast("boolean"))
        
        # startYear: STRING → INTEGER, NULL → 0
        .withColumn("start_year",
                   when(col("startYear").isNotNull(), 
                        col("startYear").cast("int"))
                   .otherwise(0))  # Replace NULL with 0
        
        # endYear: STRING → INTEGER, NULL → 9999
        .withColumn("end_year",
                   when(col("endYear").isNotNull(),
                        col("endYear").cast("int"))
                   .otherwise(9999))  # Replace NULL with 9999
        
        # runtimeMinutes: STRING → INTEGER, NULL → 0
        .withColumn("runtime_minutes",
                   when(col("runtimeMinutes").isNotNull(),
                        col("runtimeMinutes").cast("int"))
                   .otherwise(0))  # Replace NULL with 0
    )
    
    # # Add Data Quality Flags (updated logic)
    # df = (
    #     df
    #     .withColumn("has_runtime", col("runtime_minutes") > 0)  # 0 means no runtime
    #     .withColumn("has_start_year", col("start_year") > 0)  # 0 means no start year
    #     .withColumn("has_end_year", col("end_year") < 9999)  # 9999 means no end year
    #     .withColumn("has_genre", col("genres") != "Unknown")
    #     .withColumn("is_series", col("titleType").isin("tvSeries", "tvMiniSeries"))
    # )
    
    # Data Validation Flags
    df = (
        df        
        # Year range validation: start_year <= end_year
        # This will be used by expect_all_or_drop to remove invalid rows
        .withColumn("has_valid_year_range",
                   when(col("start_year") <= col("end_year"), True)
                   .otherwise(False))
        
        # Runtime validation
        .withColumn("is_reasonable_runtime",
                   when(col("runtime_minutes") == 0, None)  # No runtime data
                   .when(col("runtime_minutes") <= 1440, True)
                   .otherwise(False))
    )
    
    # Calculated Fields
    df = (
        df
        # Count genres
        .withColumn("genre_count",
                   when(col("genres") == "Unknown", 0)
                   .otherwise(size(split(col("genres"), ","))))
        
        # Series duration (only calculate if both years are real)
        # .withColumn("series_duration_years",
        #            when(col("is_series") & 
        #                 col("has_end_year") & 
        #                 col("has_start_year"),
        #                 col("end_year") - col("start_year"))
        #            .otherwise(None))
        
        # Data completeness score (0-100%)
    #     .withColumn("data_completeness_score",
    #                ((col("has_runtime").cast("int") + 
    #                  col("has_start_year").cast("int") + 
    #                  col("has_genre").cast("int")) / 3.0 * 100))
    )
    
    # Trim Text Fields
    df = (
        df
        .withColumn("primary_title", trim(col("primaryTitle")))
        .withColumn("original_title", trim(col("originalTitle")))
        .withColumn("title_type", col("titleType"))
    )
    
    # Add Silver Metadata
    df = df.withColumn("silver_processing_timestamp", current_timestamp())
    
    # FINAL COLUMN SELECTION
    return df.select(
        # Core fields (cleaned & typed)
        "tconst",
        "title_type",
        "primary_title",
        "original_title",
        "is_adult",
        "start_year",
        "end_year",
        "runtime_minutes",
        "genres",
        
        # Data quality flags
        # "has_runtime",
        # "has_start_year",
        # "has_end_year",
        # "has_genre",
        # "is_series",
        
        # Validation flags
        # "has_valid_year_range",
        # "is_reasonable_runtime",
        
        # Calculated fields
        "genre_count",
        # "series_duration_years",
        # "data_completeness_score",
        
        # Audit columns
        "ingestion_timestamp",
        "silver_processing_timestamp",
        "source_file",
        "ingestion_date"
    )

In [0]:
from pyspark.sql.functions import *
import dlt

@dlt.table(
    name="silver_title_crew",
    comment="Cleaned and exploded title crew data - one row per crew member",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_all_or_drop({
    "valid_tconst": "tconst IS NOT NULL AND tconst RLIKE '^tt[0-9]{7,8}$'",
    "crew_data_exists": "directors != 'Unknown' OR writers != 'Unknown'"
})
@dlt.expect_all_or_fail({
    "tconst_not_null": "tconst IS NOT NULL",
    "tconst_not_empty": "LENGTH(tconst) >= 9"
})
def silver_title_crew():
    """Silver transformation for title.crew with exploded crew members"""
    
    df = dlt.read_stream("bronze_title_crew_raw")
    
    # STEP 1: Replace \N with NULL
    df = df.select(
        "tconst",
        when(col("directors") == "\\N", None).otherwise(col("directors")).alias("directors"),
        when(col("writers") == "\\N", None).otherwise(col("writers")).alias("writers"),
        "ingestion_timestamp",
        "source_file",
        "ingestion_date"
    )
    
    # STEP 2: TRIM Whitespace
    df = (
        df
        .withColumn("directors", 
                   when(col("directors").isNotNull(), trim(col("directors")))
                   .otherwise(None))
        .withColumn("writers",
                   when(col("writers").isNotNull(), trim(col("writers")))
                   .otherwise(None))
    )
    
    # STEP 3: Replace NULL with "Unknown"
    df = (
        df
        .withColumn("directors",
                   when(col("directors").isNull(), "Unknown")
                   .otherwise(col("directors")))
        .withColumn("writers",
                   when(col("writers").isNull(), "Unknown")
                   .otherwise(col("writers")))
    )
    
    # STEP 4: Create arrays from comma-separated strings
    df = (
        df
        .withColumn("director_array",
                   when(col("directors") == "Unknown", array(lit("Unknown")))
                   .otherwise(split(col("directors"), ",")))
        .withColumn("writer_array",
                   when(col("writers") == "Unknown", array(lit("Unknown")))
                   .otherwise(split(col("writers"), ",")))
    )
    
    # STEP 5: Explode directors into separate rows
    df_directors = (
        df
        .select(
            "tconst",
            explode("director_array").alias("crew_member_id"),
            "ingestion_timestamp",
            "source_file",
            "ingestion_date"
        )
        .withColumn("crew_member_id", trim(col("crew_member_id")))  # Trim individual IDs
        .withColumn("crew_role", lit("director"))
    )
    
    # STEP 6: Explode writers into separate rows
    df_writers = (
        df
        .select(
            "tconst",
            explode("writer_array").alias("crew_member_id"),
            "ingestion_timestamp",
            "source_file",
            "ingestion_date"
        )
        .withColumn("crew_member_id", trim(col("crew_member_id")))  # Trim individual IDs
        .withColumn("crew_role", lit("writer"))
    )
    
    # STEP 7: Union directors and writers
    df_exploded = df_directors.union(df_writers)
    
    # STEP 8: Add data quality flags
    df_exploded = (
        df_exploded
        .withColumn("is_unknown_crew", col("crew_member_id") == "Unknown")
        .withColumn("has_valid_crew_id", 
                   col("crew_member_id").rlike("^nm[0-9]{7,8}$"))  # IMDb name format
        .withColumn("data_quality_tier",
                   when(col("is_unknown_crew"), "Unknown_Crew")
                   .when(~col("has_valid_crew_id"), "Invalid_Crew_ID")
                   .otherwise("Complete"))
    )
    
    # STEP 9: ADD SILVER METADATA
    df_exploded = df_exploded.withColumn("silver_processing_timestamp", current_timestamp())
    
    return df_exploded
